In [4]:
# Run this as Cell 1 to verify
import sqlalchemy, plotly, pandas
print(sqlalchemy.__version__)
print(plotly.__version__)

2.0.48
6.6.0


In [1]:
from sqlalchemy import create_engine
import pandas as pd
import plotly.express as px
import os

PROJECT_ID = "sg-20min-town-mod2proj"
engine = create_engine(f"bigquery://{PROJECT_ID}/olist_marts")
print("Connected!")

Connected!


In [2]:
sql = """
    SELECT
        FORMAT_DATE('%Y-%m', order_purchase_date) AS month,
        COUNT(DISTINCT order_id)                  AS order_count,
        ROUND(SUM(total_sale_amount), 2)          AS gmv,
        ROUND(AVG(total_sale_amount), 2)          AS aov
    FROM olist_marts.fact_orders
    WHERE order_status = 'delivered'
      AND order_purchase_date BETWEEN '2017-01-01' AND '2018-10-31'
    GROUP BY month
    ORDER BY month
"""
df = pd.read_sql(sql, engine)
df["gmv_mom_pct"] = df["gmv"].pct_change().mul(100).round(1)
print(f"Peak month: {df.loc[df.gmv.idxmax(), 'month']} — GMV: {df.gmv.max():,.0f}")
print(f"Avg AOV: {df.aov.mean():,.2f}")
df

Peak month: 2017-11 — GMV: 1,153,364
Avg AOV: 140.59


,month,order_count,gmv,aov,gmv_mom_pct
0,2017-01,750,127482.37,139.63,NaN
1,2017-02,1653,271239.32,145.98,112.8
2,2017-03,2546,414330.95,143.02,52.8
3,2017-04,2303,390812.40,152.13,-5.7
4,2017-05,3546,566851.40,141.57,45.0
5,2017-06,3135,490050.37,140.46,-13.5
6,2017-07,3872,566299.08,128.24,15.6
7,2017-08,4193,645832.36,134.63,14.0
8,2017-09,4150,701077.49,148.00,8.6
9,2017-10,4478,751117.01,144.06,7.1


In [3]:
fig = px.line(df, x="month", y="gmv",
    title="Monthly GMV — Olist 2017–2018",
    labels={"gmv": "GMV (BRL)", "month": "Month"},
    markers=True)
fig.update_layout(template="plotly_white")
fig.show()
fig.write_image("../docs/chart_01_monthly_gmv.png", width=1200, height=500)
print("Saved chart_01_monthly_gmv.png")

Saved chart_01_monthly_gmv.png
